# Denetim Aggregate Builder — H1 2025

**Amaç:** Panel dashboard için `denetim_agg.json` üretmek.

**Polarite kuralı:** `CEVAPDURUM` zaten doğru polarize ('Sigara içildi mi → HAYIR → Olumlu' gibi). Bu kolon doğrudan kullanılabilir.

**Ayrımlar (VARLIKTIPTANIM kolonu):**
- `Arac` → Araç denetimi (2,001,067 satır)
- `Sürücü` → Şoför denetimi (846,758 satır)
- Toplam: 2,869,509 satır

**Çıktı:** `panel_data/denetim_agg.json` — 3 varyant: tum / arac / sofor

**Şema (her varyant için):**
- `kpi`: toplam_olay, olumsuz_oturum (≥1 olumsuz cevap), olumsuz_oturum_pct, olumsuz_sayisi (soru), olumsuz_pct, benzersiz_arac, benzersiz_sicil, toplam_soru
- `tip_bazli`, `soru_bazli`, `kapino_bazli`, `sofor_bazli`

In [ ]:
import sqlite3, json, time, os
from collections import defaultdict

DB = 'panel_data/iett_data.db'
OUT = 'panel_data/denetim_agg.json'
WHERE_H1 = "TARIH BETWEEN '2025-01-01' AND '2025-06-30'"

con = sqlite3.connect(DB)
cur = con.cursor()

print('Denetim tablosu satır sayısı (H1):',
      cur.execute(f'SELECT COUNT(*) FROM denetim WHERE {WHERE_H1}').fetchone()[0])

## 1. Tek pass ile tüm aggregate'leri çıkar

In [ ]:
VARIANTS = ['tum', 'arac', 'sofor']
def init_buckets():
    return {
        'kpi': {'toplam_soru':0, 'olumsuz_sayisi':0,
                'oturum_set':set(), 'olumsuz_oturum_set':set(),
                'kapino_set':set(), 'sicil_set':set()},
        'tip': defaultdict(lambda: {'sayi':0, 'olumsuz':0}),
        'soru': defaultdict(lambda: {'sayi':0, 'olumsuz':0}),
        'kapino': defaultdict(lambda: {'sayi':0, 'olumsuz':0}),
        'sofor': defaultdict(lambda: {'sayi':0, 'olumsuz':0}),
    }
buckets = {v: init_buckets() for v in VARIANTS}

t = time.time()
processed = 0
cur.execute(f'''SELECT VARLIKTIPTANIM, DENETIMTIPTANIM, SORUACIKLAMA, CEVAPDURUM,
                       TASKID, VARLIK_KAPINO, SOFOR_SICILNO
                FROM denetim WHERE {WHERE_H1}''')
for row in cur:
    vtip, dtip, soru, cd, taskid, kapino, sicil = row
    olumsuz = 1 if cd == 'Olumsuz' else 0
    targets = ['tum']
    if vtip == 'Arac':   targets.append('arac')
    if vtip == 'Sürücü': targets.append('sofor')
    for var in targets:
        b = buckets[var]
        b['kpi']['toplam_soru'] += 1
        b['kpi']['olumsuz_sayisi'] += olumsuz
        if taskid:
            b['kpi']['oturum_set'].add(taskid)
            if olumsuz: b['kpi']['olumsuz_oturum_set'].add(taskid)
        if kapino: b['kpi']['kapino_set'].add(kapino)
        if sicil and sicil.strip() and 'Atan' not in sicil:
            b['kpi']['sicil_set'].add(sicil)
        if dtip:
            b['tip'][dtip]['sayi'] += 1
            b['tip'][dtip]['olumsuz'] += olumsuz
        if soru:
            b['soru'][soru]['sayi'] += 1
            b['soru'][soru]['olumsuz'] += olumsuz
        if kapino:
            b['kapino'][kapino]['sayi'] += 1
            b['kapino'][kapino]['olumsuz'] += olumsuz
        if sicil and sicil.strip() and 'Atan' not in sicil:
            b['sofor'][sicil]['sayi'] += 1
            b['sofor'][sicil]['olumsuz'] += olumsuz
    processed += 1
    if processed % 500000 == 0:
        print(f'  {processed:>9,} satır işlendi ({time.time()-t:.1f}s)')
print(f'BITTI {processed:,} satır, {time.time()-t:.1f}s')

## 2. JSON formata serialize et

In [ ]:
out = {}
for var, b in buckets.items():
    k = b['kpi']
    toplam_oturum = len(k['oturum_set'])
    olumsuz_oturum = len(k['olumsuz_oturum_set'])
    out[var] = {
        'kpi': {
            'toplam_olay':         toplam_oturum,
            'olumsuz_oturum':      olumsuz_oturum,
            'olumsuz_oturum_pct':  round(olumsuz_oturum/toplam_oturum*100, 1) if toplam_oturum else 0,
            'olumsuz_sayisi':      k['olumsuz_sayisi'],
            'olumsuz_pct':         round(k['olumsuz_sayisi']/k['toplam_soru']*100, 2) if k['toplam_soru'] else 0,
            'benzersiz_arac':      len(k['kapino_set']),
            'benzersiz_sicil':     len(k['sicil_set']),
            'toplam_soru':         k['toplam_soru'],
        },
        'tip_bazli': sorted([
            {'tip':t, 'sayi':d['sayi'], 'olumsuz':d['olumsuz'],
             'olumsuz_pct': round(d['olumsuz']/d['sayi']*100,1) if d['sayi'] else 0}
            for t,d in b['tip'].items()
        ], key=lambda x:-x['sayi']),
        'soru_bazli': sorted([
            {'soru':s, 'sayi':d['olumsuz'], 'total_sorulan':d['sayi'],
             'olumsuz_pct': round(d['olumsuz']/d['sayi']*100,1) if d['sayi'] else 0}
            for s,d in b['soru'].items() if d['olumsuz'] > 0
        ], key=lambda x:-x['sayi'])[:50],
        'kapino_bazli': {
            k:{'denetim_sayisi':v['sayi'], 'olumsuz':v['olumsuz'],
               'olumsuz_pct': round(v['olumsuz']/v['sayi']*100,1) if v['sayi'] else 0}
            for k,v in b['kapino'].items() if v['sayi'] >= 5
        },
        'sofor_bazli': sorted([
            {'sicil':s, 'olumsuz':d['olumsuz'], 'denetim_sayisi':d['sayi'],
             'olumsuz_pct': round(d['olumsuz']/d['sayi']*100,1) if d['sayi'] else 0}
            for s,d in b['sofor'].items() if d['olumsuz'] > 0
        ], key=lambda x:-x['olumsuz'])[:50],
    }
    print(f'\n[{var}] kpi:', out[var]['kpi'])

## 3. Yaz

In [ ]:
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, separators=(',',':'))
print(f'YAZDI {OUT} — {os.path.getsize(OUT)/1024:.0f} KB')